# Demo: section 11, calculate probability distributions from Clinton-Gore data

20260726

Author: Kyoko Kusano

This library reproduces the calculations in Section 11 of Ozawa and Khrennikov(2021), “Modeling combination of question order effect, response replicability effect, and QQ-equality with quantum instruments.”

In [1]:
from qinst_ozawa import (
    CLINTON_GORE,
    SequentialProbabilities,
    fit_independent_model,
    qq_residual,
    qqe_renormalize,
    reconstruct_jointprobdists,
    BeliefDistribution,
    PersonalityDistribution,
    IndependentModelParameters,
)

In [2]:
# read Clinton-Gore poll data: joint probablities
original = SequentialProbabilities.from_mapping(BLACK_WHITE)
BLACK_WHITE

# Or declare like this:
# original = SequentialProbabilities(
#     ay_by= 0.4899,
#     ay_bn= 0.0447,
#     an_by= 0.1767,
#     an_bn= 0.2887,
#     by_ay= 0.5625,
#     by_an= 0.1991,
#     bn_ay= 0.0255,
#     bn_an= 0.2129,
# )

{'AyBy': 0.4899,
 'AyBn': 0.0447,
 'AnBy': 0.1767,
 'AnBn': 0.2887,
 'ByAy': 0.5625,
 'ByAn': 0.1991,
 'BnAy': 0.0255,
 'BnAn': 0.2129}

In [3]:
# calculate how much this data violates QQE

print(f"{qq_residual(original)*100:.4f}%")

-0.3200%


In [4]:
# renormalize data to satisfy QQE

renormalized = qqe_renormalize(original)

probability_fields = (
    ("AyBy", "ay_by"),
    ("AyBn", "ay_bn"),
    ("AnBy", "an_by"),
    ("AnBn", "an_bn"),
    ("ByAy", "by_ay"),
    ("ByAn", "by_an"),
    ("BnAy", "bn_ay"),
    ("BnAn", "bn_an"),
)

print("Observed → QQE-renormalized")
for paper_name, field_name in probability_fields:
    original_value = getattr(renormalized.original, field_name)
    normalized_value = getattr(renormalized.normalized, field_name)
    print(
        f"{paper_name} ({field_name}): "
        f"{original_value:.4f} → {normalized_value:.4f}"
    )

print(f"\nS1: {renormalized.s1:.4f}")
print(f"S2: {renormalized.s2:.4f}")
print(
    "QQ residual after renormalization: "
    f"{qq_residual(renormalized.normalized) * 100:.4f}%"
)

Observed → QQE-renormalized
AyBy (ay_by): 0.4899 → 0.4889
AyBn (ay_bn): 0.0447 → 0.0450
AnBy (an_by): 0.1767 → 0.1780
AnBn (an_bn): 0.2887 → 0.2881
ByAy (by_ay): 0.5625 → 0.5637
ByAn (by_an): 0.1991 → 0.1977
BnAy (bn_ay): 0.0255 → 0.0253
BnAn (bn_an): 0.2129 → 0.2133

S1: 0.7770
S2: 0.2230
QQ residual after renormalization: 0.0000%


In [5]:
# estimate the parameters of the model

params = fit_independent_model(renormalized.normalized)

print("Personality distribution q(gamma)")
print(f"q0 = q(gamma=0): {params.personality.q0:.4f}")
print(f"q1 = q(gamma=1): {params.personality.q1:.4f}")
print(f"q2 = q(gamma=2): {params.personality.q2:.4f}")

print("\nBelief distribution p(A, B)")
print(f"p11 = p(A=y, B=y): {params.belief.p11:.4f}")
print(f"p10 = p(A=y, B=n): {params.belief.p10:.4f}")
print(f"p01 = p(A=n, B=y): {params.belief.p01:.4f}")
print(f"p00 = p(A=n, B=n): {params.belief.p00:.4f}")

Personality distribution q(gamma)
q0 = q(gamma=0): 0.6045
q1 = q(gamma=1): 0.0667
q2 = q(gamma=2): 0.3288

Belief distribution p(A, B)
p11 = p(A=y, B=y): 0.5184
p10 = p(A=y, B=n): 0.0155
p01 = p(A=n, B=y): 0.2430
p00 = p(A=n, B=n): 0.2231


In [6]:
# reconstruct the data from parameters

reconst = reconstruct_jointprobdists(params)

print("Observed → QQE-renormalized → Reconstructed")
for paper_name, field_name in probability_fields:
    original_value = getattr(original, field_name)
    normalized_value = getattr(renormalized.normalized, field_name)
    reconstructed_value = getattr(reconst, field_name)
    print(
        f"{paper_name} ({field_name}): "
        f"{original_value:.4f} → {normalized_value:.4f} → "
        f"{reconstructed_value:.4f}"
    )

largest_reconstruction_error = max(
    abs(
        getattr(reconst, field_name)
        - getattr(renormalized.normalized, field_name)
    )
    for _, field_name in probability_fields
)
print(
    "\nLargest difference between the QQE-renormalized and "
    f"reconstructed values: {largest_reconstruction_error:.4f}"
)

Observed → QQE-renormalized → Reconstructed
AyBy (ay_by): 0.4899 → 0.4889 → 0.4889
AyBn (ay_bn): 0.0447 → 0.0450 → 0.0450
AnBy (an_by): 0.1767 → 0.1780 → 0.1780
AnBn (an_bn): 0.2887 → 0.2881 → 0.2881
ByAy (by_ay): 0.5625 → 0.5637 → 0.5637
ByAn (by_an): 0.1991 → 0.1977 → 0.1977
BnAy (bn_ay): 0.0255 → 0.0253 → 0.0253
BnAn (bn_an): 0.2129 → 0.2133 → 0.2133

Largest difference between the QQE-renormalized and reconstructed values: 0.0000


In [12]:
# reconstruct joint probabilities from arbitary parameters

beliefs = BeliefDistribution(
    p00=0.2,
    p10=0.1,
    p01=0.4,
    p11=0.3,
)

personalities = PersonalityDistribution(
    q0=0.2,
    q1=0.4,
    q2=0.4,
)

params_a = IndependentModelParameters(
    personality = personalities,
    belief = beliefs,
)

reconstruct_jointprobdists(params_a)

SequentialProbabilities(ay_by=0.22000000000000003, ay_bn=0.18000000000000005, an_by=0.32000000000000006, an_bn=0.28, by_ay=0.33999999999999997, by_an=0.36, bn_ay=0.14, bn_an=0.16000000000000003)